In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('cars.csv')
df.head()

,brand,km_driven,fuel,owner,selling_price
0,Maruti,145500,Diesel,First Owner,450000
1,Skoda,120000,Diesel,Second Owner,370000
2,Honda,140000,Petrol,Third Owner,158000
3,Hyundai,127000,Diesel,First Owner,225000
4,Maruti,120000,Petrol,First Owner,130000


In [ ]:
pd.get_dummies(df, columns=['fuel', 'owner'])

,brand,km_driven,selling_price,fuel_CNG,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_First Owner,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,145500,450000,False,True,False,False,True,False,False,False,False
1,Skoda,120000,370000,False,True,False,False,False,False,True,False,False
2,Honda,140000,158000,False,False,False,True,False,False,False,False,True
3,Hyundai,127000,225000,False,True,False,False,True,False,False,False,False
4,Maruti,120000,130000,False,False,False,True,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
8123,Hyundai,110000,320000,False,False,False,True,True,False,False,False,False
8124,Hyundai,119000,135000,False,True,False,False,False,True,False,False,False
8125,Maruti,120000,382000,False,True,False,False,True,False,False,False,False
8126,Tata,25000,290000,False,True,False,False,True,False,False,False,False


In [ ]:
# 💡 What is happening?
# Each category becomes a new column
# Example:
# fuel    	->	fuel_Petrol	, fuel_Diesel
#owner      ->  owner_first , owner_second_owner etc.....the number of categories it consists

In [ ]:
#now to avoid the multi collinearity we drop the first column for the both generated column

In [ ]:
pd.get_dummies(df, columns=['fuel', 'owner'], drop_first=True)

,brand,km_driven,selling_price,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,145500,450000,True,False,False,False,False,False,False
1,Skoda,120000,370000,True,False,False,False,True,False,False
2,Honda,140000,158000,False,False,True,False,False,False,True
3,Hyundai,127000,225000,True,False,False,False,False,False,False
4,Maruti,120000,130000,False,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...
8123,Hyundai,110000,320000,False,False,True,False,False,False,False
8124,Hyundai,119000,135000,True,False,False,True,False,False,False
8125,Maruti,120000,382000,True,False,False,False,False,False,False
8126,Tata,25000,290000,True,False,False,False,False,False,False


In [ ]:
#now if we compare the above df's before and after dropping,
#the first columns from fuel and owner dropped i.e
# the fuel_cng from fuel column and first_owner from owner dropped

In [ ]:
# 💡 Why drop_first=True?

# If we have:

# fuel_Petrol, fuel_Diesel

# We can drop one column because:
# 👉 If Petrol = 0 → automatically Diesel = 1

# This avoids:
# ❌ multicollinearity (important in ML models)

In [ ]:
#train_test_split()

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df.iloc[:,0:4],df.iloc[:,-1],test_size=0.2,random_state=2)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

In [ ]:
from sklearn.preprocessing import OneHotEncoder
import numpy as np

ohe = OneHotEncoder(
    drop='first',           # avoid dummy variable trap
    sparse_output=False,    # ✅ NEW name (instead of sparse) here we use sparse to convert into np array, if dont use, below we have to mention explicity .toarray (to below arrays)
    dtype=np.int32          # output as integers
)

In [ ]:
X_train_new = ohe.fit_transform(X_train[['fuel','owner']]) #ohe.fit_transform(X_train[['fuel','owner']]).toarray() : if we dont set spare in above code

💡 **Important:**

    fit_transform() learns categories AND transforms
    Only used on training data

In [ ]:
X_test_new = ohe.transform(X_test[['fuel','owner']])

# 💡 **Important:**

*   Only transform() (NOT fit again!)

*   Ensures consistency




# Combine with Numerical Data

In [ ]:
np.hstack((
    X_train[['brand', 'km_driven']].values,  # original numerical columns
    X_train_new                             # encoded columns
))

array([['Hyundai', 35000, 1, ..., 0, 0, 0],
       ['Jeep', 60000, 1, ..., 0, 0, 0],
       ['Hyundai', 25000, 0, ..., 0, 0, 0],
       ...,
       ['Tata', 15000, 0, ..., 0, 0, 0],
       ['Maruti', 32500, 1, ..., 1, 0, 0],
       ['Isuzu', 121000, 1, ..., 0, 0, 0]], dtype=object)

this make sure that previously we are converted the categorical data in to numerical by taking out the columns right,
now we have to put this columns back the df with the remaining numerical data

# Count categories
now here if we see the brand there are 32 rows means it will make 32 columns so, inorder to avoid it we make the the most valued count as it is, and the for the least count we make a seperate column and add them

In [ ]:
# 💡 Why?

# Too many categories = too many columns 😵

# So we group rare brands into:
# 👉 "uncommon"

In [ ]:
counts = df['brand'].value_counts()

In [ ]:
threshold = 100
repl = counts[counts <= threshold].index

df['brand'] = df['brand'].replace(repl, 'uncommon')

# Apply Encoding

In [ ]:
pd.get_dummies(df['brand']).sample(5)

,BMW,Chevrolet,Ford,Honda,Hyundai,Mahindra,Maruti,Renault,Skoda,Tata,Toyota,Volkswagen,uncommon
6896,False,False,False,False,True,False,False,False,False,False,False,False,False
798,False,False,False,False,False,False,True,False,False,False,False,False,False
1150,False,False,False,True,False,False,False,False,False,False,False,False,False
279,False,False,False,False,False,False,False,False,False,True,False,False,False
8019,False,False,False,False,False,False,True,False,False,False,False,False,False


👉 If brands are MANY (real-world case ✅)

Do grouping:

In [ ]:
counts = df['brand'].value_counts()

threshold = 100   # you can change this

rare_brands = counts[counts < threshold].index

df['brand'] = df['brand'].replace(rare_brands, 'Other')

In [ ]:
ohe = OneHotEncoder(drop='first', sparse_output=False,handle_unknown='ignore')

In [ ]:
X_train_cat = ohe.fit_transform(X_train[['fuel', 'owner', 'brand']])
X_test_cat = ohe.transform(X_test[['fuel', 'owner', 'brand']])

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [ ]:
ohe.get_feature_names_out(['fuel', 'owner', 'brand'])

array(['fuel_Diesel', 'fuel_LPG', 'fuel_Petrol',
       'owner_Fourth & Above Owner', 'owner_Second Owner',
       'owner_Test Drive Car', 'owner_Third Owner', 'brand_Audi',
       'brand_BMW', 'brand_Chevrolet', 'brand_Daewoo', 'brand_Datsun',
       'brand_Fiat', 'brand_Force', 'brand_Ford', 'brand_Honda',
       'brand_Hyundai', 'brand_Isuzu', 'brand_Jaguar', 'brand_Jeep',
       'brand_Kia', 'brand_Land', 'brand_Lexus', 'brand_MG',
       'brand_Mahindra', 'brand_Maruti', 'brand_Mercedes-Benz',
       'brand_Mitsubishi', 'brand_Nissan', 'brand_Opel', 'brand_Peugeot',
       'brand_Renault', 'brand_Skoda', 'brand_Tata', 'brand_Toyota',
       'brand_Volkswagen', 'brand_Volvo'], dtype=object)

In [ ]:
X_train_cat_df = pd.DataFrame(
    X_train_cat,
    columns=ohe.get_feature_names_out(['fuel', 'owner', 'brand'])
)

X_test_cat_df = pd.DataFrame(
    X_test_cat,
    columns=ohe.get_feature_names_out(['fuel', 'owner', 'brand'])
)

In [ ]:
X_train_num = X_train.drop(columns=['fuel', 'owner', 'brand'])
X_test_num = X_test.drop(columns=['fuel', 'owner', 'brand'])

In [ ]:
X_train_final = pd.concat([X_train_num.reset_index(drop=True), X_train_cat_df], axis=1)
X_test_final = pd.concat([X_test_num.reset_index(drop=True), X_test_cat_df], axis=1)

In [ ]:
print(X_train_final.head())
print(X_test_final.head())

   km_driven  fuel_Diesel  fuel_LPG  fuel_Petrol  owner_Fourth & Above Owner  \
0      35000          1.0       0.0          0.0                         0.0   
1      60000          1.0       0.0          0.0                         0.0   
2      25000          0.0       0.0          1.0                         0.0   
3     130000          1.0       0.0          0.0                         0.0   
4     155000          1.0       0.0          0.0                         0.0   

   owner_Second Owner  owner_Test Drive Car  owner_Third Owner  brand_Audi  \
0                 0.0                   0.0                0.0         0.0   
1                 0.0                   0.0                0.0         0.0   
2                 0.0                   0.0                0.0         0.0   
3                 1.0                   0.0                0.0         0.0   
4                 0.0                   0.0                0.0         0.0   

   brand_BMW  ...  brand_Mitsubishi  brand_Nissan 

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train_final, y_train)

y_pred = model.predict(X_test_final)